# 01 — Data Cleaning & Preprocessing

**Goal:** Load raw CSV, inspect quality, clean text, handle edge cases, and save a clean dataset for the next phase.

**Input:** `data/raw/Customer_Sentiment.csv`  
**Output:** `data/processed/cleaned_sentiment.csv`

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print('Libraries loaded.')

## 1. Load & inspect

In [ ]:
df = pd.read_csv('../data/raw/Customer_Sentiment.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

In [ ]:
print('--- Data Types ---')
print(df.dtypes)
print()
print('--- Null Values ---')
print(df.isnull().sum())
print()
print('--- Duplicates ---')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
print('--- Categorical Value Counts ---')
cat_cols = ['gender', 'age_group', 'region', 'product_category', 'platform', 'sentiment', 'issue_resolved', 'complaint_registered']
for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts())

## 2. Clean categorical columns

In [ ]:
# Standardize text casing
str_cols = ['gender', 'age_group', 'region', 'product_category',
            'platform', 'sentiment', 'issue_resolved', 'complaint_registered']

for col in str_cols:
    df[col] = df[col].str.strip().str.lower()

# Encode binary columns as 0/1 for analysis
df['issue_resolved_bin'] = df['issue_resolved'].map({'yes': 1, 'no': 0})
df['complaint_registered_bin'] = df['complaint_registered'].map({'yes': 1, 'no': 0})

print('Categorical columns cleaned.')
df[str_cols].head(3)

## 3. Clean review_text

In [ ]:
def clean_text(text):
    """Clean raw review text for NLP processing."""
    if pd.isna(text):
        return ''
    text = str(text).lower()               # lowercase
    text = re.sub(r'http\S+', '', text)    # remove URLs
    text = re.sub(r'<.*?>', '', text)      # remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)  # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()  # normalise spaces
    return text

df['review_text_clean'] = df['review_text'].apply(clean_text)

print('Sample cleaned reviews:')
df[['review_text', 'review_text_clean']].head(5)

## 4. Feature engineering

In [ ]:
# Sentiment as numeric label (useful for correlation analysis)
sentiment_map = {'positive': 1, 'neutral': 0, 'negative': -1}
df['sentiment_score_label'] = df['sentiment'].map(sentiment_map)

# Response time buckets
df['response_time_bucket'] = pd.cut(
    df['response_time_hours'],
    bins=[0, 12, 24, 48, 72],
    labels=['<12h', '12-24h', '24-48h', '>48h']
)

print('New columns added:')
print(df[['sentiment', 'sentiment_score_label', 'response_time_hours', 'response_time_bucket']].head(8))

## 5. Save cleaned dataset

In [ ]:
df.to_csv('../data/processed/cleaned_sentiment.csv', index=False)
print(f'Saved cleaned_sentiment.csv — {len(df)} rows, {len(df.columns)} columns')
print('Columns:', df.columns.tolist())